# Analysis of Co-Design Experiments

This notebook loads result JSON files produced by `run_experiment.py` and summarizes key metrics.

In [ ]:

import json
import glob
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

results_dir = Path('results')
all_files = glob.glob(str(results_dir / '**' / '*_metrics.json'), recursive=True)
records = []
for f in all_files:
    with open(f) as fp:
        data = json.load(fp)
    data['experiment'] = Path(f).stem.replace('_metrics','')
    records.append(data)

if records:
    df = pd.DataFrame(records)
    display(df.head())
else:
    print('No result files found.')


## Summary Table (Table 1)
If result files are available, we can compute a summary table similar to Table 1 of the paper.

In [ ]:

if records:
    summary_cols = ['method', 'perplexity', 'latency', 'modularity', 'l2_cache_hit_rate']
    summary = df[summary_cols].sort_values('latency')
    display(summary)
else:
    print('Populate `results/` with experiment outputs to view summary table.')


## Modularity, Cache Hit Rate, and Latency vs Iteration (Figure 3)

In [ ]:

if records and 'iteration' in df.columns:
    iter_df = df.sort_values('iteration')
    fig, ax1 = plt.subplots()
    ax1.plot(iter_df['iteration'], iter_df['latency'], 'b-o', label='Latency (ms)')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Latency (ms)', color='b')
    ax2 = ax1.twinx()
    ax2.plot(iter_df['iteration'], iter_df['modularity'], 'orange', label='Modularity')
    ax2.plot(iter_df['iteration'], iter_df['l2_cache_hit_rate'], 'green', label='L2 Cache Hit Rate')
    ax2.set_ylabel('Score / %')
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines + lines2, labels + labels2, loc='best')
else:
    print('No iteration data available.')


## Pareto Frontier (Figure 4)

In [ ]:

if records:
    plt.figure()
    plt.scatter(df['perplexity'], df['latency'], c='red')
    for i, row in df.iterrows():
        plt.text(row['perplexity'], row['latency'], row.get('method', ''), fontsize=8)
    plt.xlabel('Perplexity')
    plt.ylabel('Latency (ms)')
    plt.title('Pareto Frontier')
    plt.gca().invert_yaxis()
    plt.show()
else:
    print('No result files found to plot Pareto frontier.')
